In [ ]:
# Load the libraries

import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt
import time

# Show PyTorch version
torch.__version__

SEED = int(time.time())

In [ ]:
# Setup device agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{device} is in use")

## Binary classification

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

In [ ]:
X_dis, y_dis = make_moons(n_samples=1000, noise=0.02, random_state=SEED)
print(X_dis.shape)
print(y_dis.shape)

In [ ]:
plt.scatter(X_dis[:,0], X_dis[:,1], s=1, c=y_dis, cmap='PiYG' )
plt.show()

In [ ]:
# converting the date into PyTorch tensors

X = torch.from_numpy(X_dis).type(torch.float)
y = torch.from_numpy(y_dis).type(torch.float)

type(X), X.dtype

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=SEED)


In [ ]:
###  model initialization

class MoonsModule_v1(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer_1 = nn.Linear(in_features=2, out_features=25)
        self.layer_2 = nn.Linear(in_features=25, out_features=25)
        self.layer_3 = nn.Linear(in_features=25, out_features=25)
        self.layer_4 = nn.Linear(in_features=25, out_features=1)
        self.relu    = nn.ReLU()
        self.tahn    = nn.Tanh()


    def forward(self, x):
        return self.layer_4(  self.relu(  self.layer_3( self.tahn( self.layer_2( self.relu( self.layer_1(x) ) ) ) ) ) )

In [ ]:
model_1 = MoonsModule_v1().to(device)

In [ ]:
# model_1.state_dict()

In [ ]:
def acc_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = 100*correct/len(y_pred)
    return acc

In [ ]:

### setting up the training

torch.manual_seed( SEED )
torch.cuda.manual_seed( SEED )

epochs = 10

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(params=model_1.parameters(),
                            lr=0.05)

X_train, y_train = X_train.to(device), y_train.to(device)
X_test,  y_test  = X_test.to(device),  y_test.to(device)


for epoch in range(epochs):

    model_1.train()

    y_logits = model_1(X_train).squeeze()
    y_pred = torch.round(torch.sigmoid(y_logits))

    loss = loss_fn(y_logits, y_train)
    acc = acc_fn(y_train, y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ### Testing
    model_1.eval()

    with torch.inference_mode():

        test_logits = model_1(X_test).squeeze()
        test_pred = torch.round( torch.sigmoid(test_logits) )

        test_loss = loss_fn(test_logits, y_test)
        test_acc = acc_fn(y_test, test_pred)

        if epoch > 1 and epoch%10 == 0:
            print(f"Epoch: {epoch} | Loss: {loss:.5f}, Acc: {acc:.2f}% | Test Loss: {test_loss:.5f}, Test Acc: {test_acc:.2f}%")   

In [ ]:
import requests
from pathlib import Path

# Download helper functions from Learn PyTorch repo
if Path("helper_functions.py").is_file():
    print("helper_functions.py already exists, skipping downloading")
else:
  print("Downloading helper_functions.py")
  request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
  with open("helper_functions.py", "wb") as f:
    f.write(request.content)

from helper_functions import plot_predictions, plot_decision_boundary

In [ ]:
# Plot decision boundaries for training and test sets
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Train")
plot_decision_boundary(model_1, X_train, y_train)
plt.subplot(1, 2, 2)
plt.title("Test")
plot_decision_boundary(model_1, X_test, y_test)

### Multiclass classification

In [ ]:
# Code for creating a spiral dataset from CS231n
import numpy as np
N = 300 # number of points per class
D = 2 # dimensionality
K = 3 # number of classes
X = np.zeros((N*K,D)) # data matrix (each row = single example)
y = np.zeros(N*K, dtype='uint8') # class labels
for j in range(K):
  ix = range(N*j,N*(j+1))
  r = np.linspace(0.0,1,N) # radius
  t = np.linspace(j*4,(j+1)*4,N) + np.random.randn(N)*0.2 # theta
  X[ix] = np.c_[r*np.sin(t), r*np.cos(t)]
  y[ix] = j
# lets visualize the data
plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.Spectral)
plt.show()

In [ ]:
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.LongTensor)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=SEED)

In [ ]:
###  model initialization

class SpiralModule_v1(nn.Module):

    def __init__(self, input_features, hidden_layers, output_features):
        super().__init__()

        self.linear_layers_stack = nn.Sequential(
            nn.Linear(in_features=input_features, out_features=hidden_layers),
            nn.Tanh(),
            nn.Linear(in_features=hidden_layers, out_features=hidden_layers),
            nn.ReLU(),
            nn.Linear(in_features=hidden_layers, out_features=output_features)
        )


    def forward(self, x):
        return self.linear_layers_stack(x)

In [ ]:
model_2 = SpiralModule_v1(input_features=2, 
                          hidden_layers=10, 
                          output_features=3).to(device)

In [ ]:
# model_2.state_dict()

In [ ]:
### setting up the training

torch.manual_seed( SEED )
torch.cuda.manual_seed( SEED )

epochs = 100

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_2.parameters(),
                            lr=0.1)

X_train, y_train = X_train.to(device), y_train.to(device)
X_test,  y_test  = X_test.to(device),  y_test.to(device)


for epoch in range(epochs+1):

    model_2.train()

    y_logits = model_2(X_train)
    y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)

    loss = loss_fn(y_logits, y_train)
    acc = acc_fn(y_train, y_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ### Testing
    model_2.eval()

    with torch.inference_mode():

        test_logits = model_2(X_test).squeeze()
        test_pred =  torch.softmax(test_logits, dim=1).argmax(dim=1)

        test_loss = loss_fn(test_logits, y_test)
        test_acc = acc_fn(y_test, test_pred)

        if epoch > 1 and epoch%10 == 0:
            print(f"Epoch: {epoch} | Loss: {loss:.5f}, Acc: {acc:.2f}% | Test Loss: {test_loss:.5f}, Test Acc: {test_acc:.2f}%")   

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Train")
plot_decision_boundary(model_2, X_train, y_train)
plt.subplot(1, 2, 2)
plt.title("Test")
plot_decision_boundary(model_2, X_test, y_test)